
# Zero-shot Benchmark — **DINOv2-L** (+LinearProbe)

Notebook coerente con le versioni precedenti: stessa pipeline, metriche e salvataggi.
Pensato per **Colab + Google Drive** sul task di *architectural style classification*.

### Cosa fa
- Usa un backbone pre-addestrato (ViT “DINO”) per estrarre feature fisse dalle immagini
- Poi allena una Logistic Regression multinomiale sopra a quelle feature per classificare.
- Stampa: Accuracy, Macro-F1, Balanced Accuracy, classification_report per classe.
- Calcola Top-k accuracy (k = 1,2,3,5) usando le probabilità.
- Disegna la confusion matrix normalizzata (matplotlib).
Latenza
- Prende un batch dal test, fa un forward pass del backbone e misura la latenza media per immagine (ms/img).
- Appende i risultati a `benchmark_results.csv` (in `/content` e nella cartella del dataset in Drive)

> Modifica i parametri nella sezione **CONFIG** qui sotto.


In [ ]:
#versione -----22/08
# ====================================================
# DINOv2-B + Linear Probe (versione otimizzata per Colab)
# ====================================================

!pip -q install timm scikit-learn matplotlib pandas joblib

import os, time, glob, json, shutil
import numpy as np
import pandas as pd
import torch
import timm
import matplotlib.pyplot as plt

from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
from torchvision import transforms

from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    confusion_matrix, top_k_accuracy_score, balanced_accuracy_score
)
import joblib

# ----------------------------
# MOUNT DRIVE
# ----------------------------
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# ----------------------------
# CONFIG (modifica qui se serve)
# ----------------------------
# Sorgente dataset su Drive (metti il tuo path giusto qui):
DRIVE_DATA_ROOT = "/content/drive/MyDrive/ArchiStyles-2"   # deve contenere train/ valid/ test
LOCAL_DATA_ROOT = "/content/ArchiStyles-2"                 # destinazione locale veloce
TRAIN_SPLIT, VAL_SPLIT, TEST_SPLIT = "train", "valid", "test"

# Batch & loader
BATCH_SIZE = 64
NUM_WORKERS = max(2, (os.cpu_count() or 4)//2)
PIN_MEMORY = True

# Modello backbone
BACKBONE_NAME = "vit_base_patch16_224.dino"
IMG_SIZE = 224
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Probe & ricerca iperparametri
USE_RIDGE = False          # True -> usa RidgeClassifier (molto rapido, niente predict_proba)
LOGREG_SOLVER = "lbfgs"    # "lbfgs" consigliato per multinomial su feature scalate
LOGREG_MAX_ITER = 600
LOGREG_TOL = 1e-3
C_GRID = [0.1, 1.0]        # griglia snella (veloce)

# Percorsi output persistenti su Drive
OUT_DIR = os.path.join(DRIVE_DATA_ROOT, "outputs")
FEAT_DIR = os.path.join(OUT_DIR, "features")
ART_DIR  = os.path.join(OUT_DIR, "artifacts")
os.makedirs(FEAT_DIR, exist_ok=True)
os.makedirs(ART_DIR, exist_ok=True)

SCORECARD_CSV = os.path.join(OUT_DIR, "benchmark_results.csv")

# ----------------------------
# COPIA DATASET IN LOCALE (una volta per sessione)
# ----------------------------
def copy_dataset_to_local(src, dst):
    if not all(os.path.isdir(os.path.join(src, d)) for d in [TRAIN_SPLIT, VAL_SPLIT, TEST_SPLIT]):
        raise FileNotFoundError(f"Dataset non trovato in {src}. Devono esistere {TRAIN_SPLIT}/ {VAL_SPLIT}/ {TEST_SPLIT}/")
    if os.path.isdir(dst) and all(os.path.isdir(os.path.join(dst, d)) for d in [TRAIN_SPLIT, VAL_SPLIT, TEST_SPLIT]):
        print("[INFO] Dataset già presente in locale:", dst)
        return
    print("[INFO] Copia dataset su disco locale (prima volta della sessione)...")
    # rsync è robusto su tante piccole file operations
    !rsync -ah --info=progress2 "{src}/" "{dst}/"
    print("[OK] Copiato in:", dst)

copy_dataset_to_local(DRIVE_DATA_ROOT, LOCAL_DATA_ROOT)

# ----------------------------
# DATASET & DATALOADER
# ----------------------------
normalize = transforms.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225))
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    normalize,
])

train_ds = ImageFolder(os.path.join(LOCAL_DATA_ROOT, TRAIN_SPLIT), transform=transform)
val_ds   = ImageFolder(os.path.join(LOCAL_DATA_ROOT, VAL_SPLIT),   transform=transform)
test_ds  = ImageFolder(os.path.join(LOCAL_DATA_ROOT, TEST_SPLIT),  transform=transform)

classnames = train_ds.classes
print(f"[INFO] Classi: {len(classnames)}")

train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
val_dl   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
test_dl  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

# ----------------------------
# BACKBONE: DINOv2-B (solo feature, no head)
# ----------------------------
model = timm.create_model(BACKBONE_NAME, pretrained=True, num_classes=0)
model = model.to(DEVICE).eval()

@torch.no_grad()
def extract_features(dataloader):
    feats, labels = [], []
    for imgs, lbls in dataloader:
        imgs = imgs.to(DEVICE, non_blocking=True)
        f = model(imgs).cpu().numpy()
        feats.append(f); labels.append(lbls.numpy())
    return np.concatenate(feats), np.concatenate(labels)

# ----------------------------
# CACHE FEATURE: carica se esiste, altrimenti estrai & salva
# ----------------------------
feat_paths = {
    "X_train": os.path.join(FEAT_DIR, "X_train.npy"),
    "y_train": os.path.join(FEAT_DIR, "y_train.npy"),
    "X_val":   os.path.join(FEAT_DIR, "X_val.npy"),
    "y_val":   os.path.join(FEAT_DIR, "y_val.npy"),
    "X_test":  os.path.join(FEAT_DIR, "X_test.npy"),
    "y_test":  os.path.join(FEAT_DIR, "y_test.npy"),
}

def features_exist(paths):
    return all(os.path.exists(p) for p in paths.values())

start_gpu = time.time()
if features_exist(feat_paths):
    print("[INFO] Carico feature cache da Drive...")
    X_train = np.load(feat_paths["X_train"]); y_train = np.load(feat_paths["y_train"])
    X_val   = np.load(feat_paths["X_val"]);   y_val   = np.load(feat_paths["y_val"])
    X_test  = np.load(feat_paths["X_test"]);  y_test  = np.load(feat_paths["y_test"])
else:
    print("[INFO] Estraggo feature con DINOv2-B (GPU)...")
    X_train, y_train = extract_features(train_dl)
    X_val,   y_val   = extract_features(val_dl)
    X_test,  y_test  = extract_features(test_dl)
    # salva cache
    np.save(feat_paths["X_train"], X_train); np.save(feat_paths["y_train"], y_train)
    np.save(feat_paths["X_val"],   X_val);   np.save(feat_paths["y_val"],   y_val)
    np.save(feat_paths["X_test"],  X_test);  np.save(feat_paths["y_test"],  y_test)
    print("[OK] Feature salvate in", FEAT_DIR)
gpu_time_h = (time.time() - start_gpu) / 3600
print(f"[INFO] Tempo GPU-h ≈ {gpu_time_h:.3f}")

# ----------------------------
# SCALER
# ----------------------------
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_val_sc   = scaler.transform(X_val)
X_test_sc  = scaler.transform(X_test)

# ----------------------------
# PROBE: Logistic Regression (veloce) o Ridge
# ----------------------------
if USE_RIDGE:
    print("[INFO] Uso RidgeClassifier (molto rapido)")
    # una sola fit, niente prob
    clf = RidgeClassifier(alpha=1.0)
    clf.fit(X_train_sc, y_train)
    best_C = None
else:
    print("[INFO] Uso LogisticRegression (solver=lbfgs) con grid ristretta su C", C_GRID)
    best_acc, best_C = -1.0, None
    for C in C_GRID:
        tmp = LogisticRegression(
            max_iter=LOGREG_MAX_ITER, C=C, solver=LOGREG_SOLVER,
            multi_class="multinomial", tol=LOGREG_TOL, n_jobs=None
        )
        tmp.fit(X_train_sc, y_train)
        acc = accuracy_score(y_val, tmp.predict(X_val_sc))
        print(f"  C={C}: Val Acc={acc:.4f}")
        if acc > best_acc:
            best_acc, best_C = acc, C
    clf = LogisticRegression(
        max_iter=LOGREG_MAX_ITER, C=best_C, solver=LOGREG_SOLVER,
        multi_class="multinomial", tol=LOGREG_TOL, n_jobs=None
    )
    clf.fit(X_train_sc, y_train)

# ----------------------------
# TEST & METRICHE
# ----------------------------
y_pred = clf.predict(X_test_sc)
acc = accuracy_score(y_test, y_pred)
macro_f1 = f1_score(y_test, y_pred, average="macro")
bal_acc = balanced_accuracy_score(y_test, y_pred)

print("\n=== TEST ===")
print("Accuracy:", acc)
print("Macro-F1:", macro_f1)
print("Balanced Acc:", bal_acc)
print(classification_report(y_test, y_pred, target_names=classnames, digits=4, zero_division=0))

probs = None
top5 = None
if not USE_RIDGE and hasattr(clf, "predict_proba"):
    probs = clf.predict_proba(X_test_sc)
    for k in [1, 2, 3, 5]:
        print(f"Top-{k} Accuracy:", top_k_accuracy_score(y_test, probs, k=k, labels=range(len(classnames))))
    top5 = top_k_accuracy_score(y_test, probs, k=5, labels=range(len(classnames)))

cm = confusion_matrix(y_test, y_pred, normalize="true")
plt.figure(figsize=(8,6))
plt.imshow(cm, cmap="Blues"); plt.title("Confusion Matrix (normalized)")
plt.colorbar(); plt.tight_layout()
plt.show()

# ----------------------------
# LATENZA BACKBONE
# ----------------------------
imgs, _ = next(iter(test_dl))
imgs = imgs.to(DEVICE, non_blocking=True)
start = time.time()
with torch.no_grad():
    _ = model(imgs)
latency = (time.time() - start) / imgs.size(0) * 1000
print(f"[INFO] Backbone latency: {latency:.2f} ms/img")

# ----------------------------
# SALVATAGGI ARTEFATTI & SCORECARD
# ----------------------------
# salva scaler, modello, classi, predizioni
joblib.dump(scaler, os.path.join(ART_DIR, "scaler.pkl"))
joblib.dump(clf,    os.path.join(ART_DIR, "probe.pkl"))
with open(os.path.join(ART_DIR, "classes.json"), "w") as f:
    json.dump(classnames, f)

np.save(os.path.join(OUT_DIR, "y_test.npy"), y_test)
np.save(os.path.join(OUT_DIR, "y_pred.npy"), y_pred)
if probs is not None:
    np.save(os.path.join(OUT_DIR, "probs.npy"), probs)

# scorecard
row = {
    "Model": f"DINOv2-B + {'Ridge' if USE_RIDGE else 'LogReg(lbfgs)'}",
    "Top-1 Acc": float(acc),
    "Macro-F1": float(macro_f1),
    "Top-5 Acc": float(top5) if top5 is not None else np.nan,
    "Balanced Acc": float(bal_acc),
    "GPU-h": float(gpu_time_h),
    "Latency (ms/img)": float(latency),
    "Effort umano (h)": "~3",
    "Note": f"Linear probe su feature {BACKBONE_NAME}; dataset locale; cache feature; solver {'ridge' if USE_RIDGE else LOGREG_SOLVER}; C={best_C if not USE_RIDGE else 'n/a'}",
}
df = pd.DataFrame([row])

if os.path.exists(SCORECARD_CSV):
    df.to_csv(SCORECARD_CSV, mode="a", header=False, index=False)
else:
    df.to_csv(SCORECARD_CSV, index=False)

print("\n[OK] Artefatti salvati in:", ART_DIR)
print("[OK] Feature/Predizioni in:", OUT_DIR)
print("[OK] Scorecard aggiornata:", SCORECARD_CSV)
print(df)

